# Sleep Staging — 12: Robustez entre semillas

Implementa las sugerencias de Facundo del 12/10/2026. Cinco semillas predefinidas; ventanas 3, 5 y 7 con los mismos centros y split. Se reutiliza una única CNN y sus embeddings: la variabilidad estimada es condicional a ese encoder, no la del pipeline completo. Media ± desvío muestral (ddof=1) no es un intervalo de confianza.

El contexto se identifica por media de validation antes de evaluar test. No se elige la mejor semilla. El fine-tuning compara cada modelo de contexto 5 con su propio checkpoint congelado. Se conserva la decisión histórica y se reporta esta revisión como análisis de robustez.

**Para continuar sin reentrenar:** las secciones originales ya están ejecutadas y se conservan como registro. Reiniciá el kernel y ejecutá **únicamente la celda de código bajo “Evaluación final de contexto 7 — sin entrenamiento”**, al final. No uses “Run All”: las celdas anteriores contienen los entrenamientos originales. Ejecutar desde la carpeta `Notebooks/` del proyecto. Se requieren los parquet y CSV ya generados en `reports/results` y `reports/results/robustness`.


In [2]:
from pathlib import Path
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../reports/results")
FIGURES_DIR = Path("../reports/figures")
MODELS_DIR = Path("../models")

for directory in (RESULTS_DIR, FIGURES_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Artefactos generados previamente
EMBEDDINGS_PATH = DATA_DIR / "dl_cnn_embeddings.npy"
CACHE_META_PATH = DATA_DIR / "dl_metadata.parquet"
DL_SPLIT_PATH = RESULTS_DIR / "dl_subject_split.json"

# Salidas de NB9
VALIDATION_SUMMARY_PATH = RESULTS_DIR / "dl_context_size_validation.csv"
TEST_SUMMARY_PATH = RESULTS_DIR / "dl_context_size_test.csv"
CLASS_RESULTS_PATH = RESULTS_DIR / "dl_context_size_per_class.csv"
TRANSITION_RESULTS_PATH = RESULTS_DIR / "dl_context_size_transition.csv"
FIGURE_PATH = FIGURES_DIR / "dl_context_size_macro_f1.png"

STAGE_MAP = {0: "W", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
STAGE_ORDER = list(STAGE_MAP)
STAGE_NAMES = [STAGE_MAP[idx] for idx in STAGE_ORDER]

CONTEXT_SIZES = [3, 5, 7]
EMBEDDING_DIM = 128
HIDDEN_DIM = 64

BATCH_SIZE = 256
MAX_EPOCHS = 25
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
SEEDS = [42, 123, 2026, 7, 99]
# True ejecuta también la parte costosa sobre señales originales.
RUN_FINETUNING = True
RESULTS_DIR = RESULTS_DIR / "robustness"
MODELS_DIR = MODELS_DIR / "robustness"
for p in (RESULTS_DIR, MODELS_DIR): p.mkdir(parents=True, exist_ok=True)


PyTorch: 2.11.0+cu128
Device:  cuda
GPU:     NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:
required_paths = [
    EMBEDDINGS_PATH,
    CACHE_META_PATH,
    DL_SPLIT_PATH,
]

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos necesarios. Ejecutá NB8 antes de NB9:\n"
        + "\n".join(f"- {path}" for path in missing)
    )

embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
metadata = pd.read_parquet(CACHE_META_PATH).reset_index(drop=True)

if len(embeddings) != len(metadata):
    raise ValueError("Embeddings y metadata tienen distinto número de epochs.")

with open(DL_SPLIT_PATH, "r", encoding="utf-8") as file:
    split_info = json.load(file)

train_subjects = set(split_info["train_subjects"])
val_subjects = set(split_info["validation_subjects"])
test_subjects = set(split_info["test_subjects"])

assert train_subjects.isdisjoint(val_subjects)
assert train_subjects.isdisjoint(test_subjects)
assert val_subjects.isdisjoint(test_subjects)

print(f"Embeddings: {embeddings.shape}")
print(f"Sujetos:    {metadata['subject'].nunique()}")
print(f"Train:      {len(train_subjects)} sujetos")
print(f"Validation: {len(val_subjects)} sujetos")
print(f"Test:       {len(test_subjects)} sujetos")
if not np.array_equal(metadata["epoch_global"].to_numpy(), np.arange(len(metadata))):
    raise ValueError("epoch_global debe corresponder a la fila del cache.")


Embeddings: (135471, 128)
Sujetos:    149
Train:      95 sujetos
Validation: 24 sujetos
Test:       30 sujetos


In [4]:
def build_temporal_index(metadata, context_size):
    half = context_size // 2
    rows = []

    for subject, subject_df in metadata.groupby("subject", sort=True):
        subject_df = subject_df.sort_values("epoch_global").reset_index(drop=True)

        global_idx = subject_df["epoch_global"].to_numpy(dtype=np.int64)
        source_idx = subject_df["source_epoch_index"].to_numpy(dtype=np.int64)
        onset = subject_df["recording_onset_s"].to_numpy(dtype=np.float64)
        labels = subject_df["label"].to_numpy(dtype=np.int64)

        for center in range(half, len(subject_df) - half):
            start = center - half
            stop = center + half + 1

            source_window = source_idx[start:stop]
            onset_window = onset[start:stop]

            if not np.all(np.diff(source_window) == 1):
                continue

            if not np.allclose(np.diff(onset_window), 30.0, atol=1e-3):
                continue

            rows.append({
                "subject": subject,
                "source_epoch_index": int(source_idx[center]),
                "center_epoch_global": int(global_idx[center]),
                "label": int(labels[center]),
                "sequence_indices": global_idx[start:stop].tolist(),
                "is_transition": bool(
                    labels[center - 1] != labels[center]
                    or labels[center + 1] != labels[center]
                ),
            })

    return pd.DataFrame(rows)


full_indexes = {
    size: build_temporal_index(metadata, size)
    for size in CONTEXT_SIZES
}

for size, index_df in full_indexes.items():
    print(f"Contexto {size}: {len(index_df):,} secuencias válidas")

Contexto 3: 135,168 secuencias válidas
Contexto 5: 134,866 secuencias válidas
Contexto 7: 134,564 secuencias válidas


In [5]:
MAX_CONTEXT = max(CONTEXT_SIZES)
join_cols = ["subject", "source_epoch_index"]

# Los centros válidos para 7 epochs forman el conjunto común.
common_keys = full_indexes[MAX_CONTEXT][join_cols].copy()

indexes = {}

for size in CONTEXT_SIZES:
    index_df = common_keys.merge(
        full_indexes[size],
        on=join_cols,
        how="left",
        validate="one_to_one",
    )

    if index_df["sequence_indices"].isna().any():
        raise ValueError(
            f"Hay centros del conjunto común sin ventana válida para contexto {size}."
        )

    indexes[size] = index_df

summary_rows = []

for size in CONTEXT_SIZES:
    index_df = indexes[size]

    summary_rows.append({
        "context_size": size,
        "seconds": size * 30,
        "train": index_df["subject"].isin(train_subjects).sum(),
        "validation": index_df["subject"].isin(val_subjects).sum(),
        "test": index_df["subject"].isin(test_subjects).sum(),
    })

sequence_summary = pd.DataFrame(summary_rows)
display(sequence_summary)

assert sequence_summary[["train", "validation", "test"]].nunique().eq(1).all()

,context_size,seconds,train,validation,test
0,3,90,86730,21388,26446
1,5,150,86730,21388,26446
2,7,210,86730,21388,26446


In [6]:
class TemporalEmbeddingDataset(Dataset):
    def __init__(self, embeddings, index_df):
        self.embeddings = embeddings
        self.sequence_indices = np.stack(
            index_df["sequence_indices"].apply(np.asarray).to_numpy()
        ).astype(np.int64)
        self.labels = index_df["label"].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, item):
        indices = self.sequence_indices[item]
        x = np.asarray(self.embeddings[indices], dtype=np.float32)
        y = self.labels[item]

        return (
            torch.from_numpy(x),
            torch.tensor(y, dtype=torch.long),
        )


class TemporalBiGRU(nn.Module):
    def __init__(self, input_dim=128, hidden_dim=64, n_classes=5):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(0.30)
        self.classifier = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x):
        sequence_output, _ = self.gru(x)
        center = x.shape[1] // 2
        center_output = sequence_output[:, center, :]

        return self.classifier(self.dropout(center_output))


def partition_index(index_df, subjects):
    return index_df[
        index_df["subject"].isin(subjects)
    ].reset_index(drop=True)

In [7]:
# Como los centros son los mismos para los tres experimentos,
# los pesos de clase también pueden calcularse una sola vez.
reference_train = partition_index(
    indexes[MAX_CONTEXT],
    train_subjects,
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.asarray(STAGE_ORDER),
    y=reference_train["label"].to_numpy(),
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE,
)

weights_df = pd.DataFrame({
    "stage": STAGE_NAMES,
    "weight": class_weights.cpu().numpy(),
})

display(weights_df.round(3))

,stage,weight
0,W,1.139
1,N1,1.812
2,N2,0.548
3,N3,1.036
4,REM,1.282


In [8]:
def reset_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loaders(index_df, seed=RANDOM_STATE):
    train_index = partition_index(index_df, train_subjects)
    val_index = partition_index(index_df, val_subjects)
    test_index = partition_index(index_df, test_subjects)

    train_dataset = TemporalEmbeddingDataset(embeddings, train_index)
    val_dataset = TemporalEmbeddingDataset(embeddings, val_index)
    test_dataset = TemporalEmbeddingDataset(embeddings, test_index)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    return (
        train_index,
        val_index,
        test_index,
        train_loader,
        val_loader,
        test_loader,
    )


def predict_loader(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            pred = logits.argmax(dim=1)

            y_true.append(yb.numpy())
            y_pred.append(pred.cpu().numpy())

    return np.concatenate(y_true), np.concatenate(y_pred)


def evaluate_loader(model, loader):
    y_true, y_pred = predict_loader(model, loader)

    return {
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=STAGE_ORDER),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=STAGE_ORDER, zero_division=0),
    }

In [9]:
def train_context_model(context_size, index_df, seed):
    reset_seed(seed)

    (
        train_index,
        val_index,
        test_index,
        train_loader,
        val_loader,
        test_loader,
    ) = make_loaders(index_df, seed)

    model = TemporalBiGRU(
        input_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        n_classes=len(STAGE_NAMES),
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    model_path = MODELS_DIR / f"dl_context_{context_size}_seed_{seed}_bigru.pt"
    history_path = (
        RESULTS_DIR / f"dl_context_{context_size}_seed_{seed}_training_history.csv"
    )

    history = []
    best_macro_f1 = -np.inf
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        running_loss = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(yb)
            n_train += len(yb)

        train_loss = running_loss / n_train
        val_metrics = evaluate_loader(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_kappa": val_metrics["cohen_kappa"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })

        print(
            f"Contexto {context_size} | "
            f"epoch {epoch:02d} | "
            f"loss={train_loss:.4f} | "
            f"val Macro-F1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(model.state_dict(), model_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping.")
            break

    history = pd.DataFrame(history)
    history.to_csv(history_path, index=False)

    model.load_state_dict(
        torch.load(model_path, map_location=DEVICE)
    )

    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "test_index": test_index,
        "test_loader": test_loader,
    }

In [10]:
validation_rows = []
for size in CONTEXT_SIZES:
    for seed in SEEDS:
        result = train_context_model(size, indexes[size], seed)
        best = result["history"].loc[result["history"]["val_macro_f1"].idxmax()]
        validation_rows.append({"context_size": size, "seed": seed, **best.to_dict()})
        pd.DataFrame(validation_rows).to_csv(RESULTS_DIR / "validation_runs.csv", index=False)
        del result
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
validation_runs = pd.DataFrame(validation_rows)
validation_summary = validation_runs.groupby("context_size")[["val_macro_f1", "val_kappa", "val_accuracy", "val_balanced_accuracy"]].agg(["mean", "std"])
validation_summary.to_csv(RESULTS_DIR / "validation_summary.csv")
display(validation_summary)
selected_context = int(validation_summary[("val_macro_f1", "mean")].idxmax())
(RESULTS_DIR / "validation_selection.json").write_text(json.dumps({"context_size": selected_context, "seeds": SEEDS, "criterion": "mean_validation_macro_f1", "encoder": "fixed_NB6", "changes_historical_selection": False}, indent=2))
print("Mayor media de validation:", selected_context)
paired_windows = validation_runs.pivot(index="seed", columns="context_size", values="val_macro_f1")
deltas = pd.DataFrame({"5_minus_3": paired_windows[5]-paired_windows[3], "7_minus_5": paired_windows[7]-paired_windows[5]})
deltas.to_csv(RESULTS_DIR / "window_paired_deltas.csv")
display(deltas.agg(["mean", "std"]))


Contexto 3 | epoch 01 | loss=0.4805 | val Macro-F1=0.7682
Contexto 3 | epoch 02 | loss=0.4046 | val Macro-F1=0.7599
Contexto 3 | epoch 03 | loss=0.3985 | val Macro-F1=0.7602
Contexto 3 | epoch 04 | loss=0.3956 | val Macro-F1=0.7625
Contexto 3 | epoch 05 | loss=0.3937 | val Macro-F1=0.7687
Contexto 3 | epoch 06 | loss=0.3907 | val Macro-F1=0.7691
Contexto 3 | epoch 07 | loss=0.3888 | val Macro-F1=0.7630
Contexto 3 | epoch 08 | loss=0.3876 | val Macro-F1=0.7600
Contexto 3 | epoch 09 | loss=0.3859 | val Macro-F1=0.7632
Contexto 3 | epoch 10 | loss=0.3856 | val Macro-F1=0.7718
Contexto 3 | epoch 11 | loss=0.3849 | val Macro-F1=0.7642
Contexto 3 | epoch 12 | loss=0.3847 | val Macro-F1=0.7700
Contexto 3 | epoch 13 | loss=0.3837 | val Macro-F1=0.7755
Contexto 3 | epoch 14 | loss=0.3822 | val Macro-F1=0.7742
Contexto 3 | epoch 15 | loss=0.3819 | val Macro-F1=0.7663
Contexto 3 | epoch 16 | loss=0.3811 | val Macro-F1=0.7680
Contexto 3 | epoch 17 | loss=0.3802 | val Macro-F1=0.7746
Contexto 3 | e

val_macro_f1           val_kappa           val_accuracy  \
                     mean       std      mean       std         mean   
context_size                                                           
3                0.774055  0.002479  0.728285  0.004678     0.790453   
5                0.778815  0.002640  0.732207  0.003086     0.793445   
7                0.781434  0.001672  0.735453  0.002600     0.796157   

                       val_balanced_accuracy            
                   std                  mean       std  
context_size                                            
3             0.003986              0.784920  0.001909  
5             0.002717              0.789850  0.001370  
7             0.002079              0.791555  0.002268

Mayor media de validation: 7


,5_minus_3,7_minus_5
mean,0.004759,0.002619
std,0.001022,0.003472


## Test de las configuraciones predefinidas
Se evalúan todas las semillas, sin seleccionar por test. Las diferencias entre semillas no representan nuevos sujetos independientes.

In [11]:
test_rows = []
for size in CONTEXT_SIZES:
    for seed in SEEDS:
        model = TemporalBiGRU().to(DEVICE)
        model.load_state_dict(torch.load(MODELS_DIR / f"dl_context_{size}_seed_{seed}_bigru.pt", map_location=DEVICE))
        *_, test_index, train_loader, val_loader, test_loader = make_loaders(indexes[size], seed)
        test_rows.append({"context_size": size, "seed": seed, **evaluate_loader(model, test_loader)})
        y_true, y_pred = predict_loader(model, test_loader)
        pred = test_index[["subject", "source_epoch_index", "center_epoch_global"]].copy()
        pred["y_true"], pred["y_pred"] = y_true, y_pred
        pred.to_parquet(RESULTS_DIR / f"context_{size}_seed_{seed}_predictions.parquet", index=False)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
test_runs = pd.DataFrame(test_rows)
test_runs.to_csv(RESULTS_DIR / "test_runs.csv", index=False)
test_summary = test_runs.groupby("context_size")[["macro_f1", "cohen_kappa", "accuracy", "balanced_accuracy"]].agg(["mean", "std"])
test_summary.to_csv(RESULTS_DIR / "test_summary.csv")
display(test_summary)


macro_f1           cohen_kappa            accuracy            \
                  mean       std        mean       std      mean       std   
context_size                                                                 
3             0.766665  0.002332    0.716273  0.004682  0.781464  0.004063   
5             0.769603  0.003285    0.719579  0.004015  0.784134  0.003484   
7             0.774928  0.001841    0.725698  0.003195  0.789102  0.002489   

             balanced_accuracy            
                          mean       std  
context_size                              
3                     0.782521  0.001946  
5                     0.784574  0.003433  
7                     0.788695  0.003377

## Fine-tuning pareado, contexto 5
Usa cada checkpoint congelado de la sección anterior y el encoder original. Reinicia optimizador y orden de batches por semilla. Requiere dl_signals.npy, normalización y dl_cnn_baseline.pt. Puede omitirse con RUN_FINETUNING=False, pero en ese caso la robustez del fine-tuning queda pendiente.

In [12]:
class TemporalSignalDataset(Dataset):
    def __init__(self, signals, index_df, mean, std):
        self.signals = signals

        self.sequence_indices = np.stack(
            index_df["sequence_indices"]
            .apply(np.asarray)
            .to_numpy()
        ).astype(np.int64)

        self.labels = index_df[
            "label"
        ].to_numpy(dtype=np.int64)

        self.mean = np.asarray(
            mean,
            dtype=np.float32,
        )[None, :, None]

        self.std = np.asarray(
            std,
            dtype=np.float32,
        )[None, :, None]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, item):
        indices = self.sequence_indices[item]

        x = np.asarray(
            self.signals[indices],
            dtype=np.float32,
        )

        x = (x - self.mean) / self.std
        y = self.labels[item]

        return (
            torch.from_numpy(x),
            torch.tensor(y, dtype=torch.long),
        )


class SleepCNN(nn.Module):
    def __init__(self, n_channels=7, n_classes=5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, n_classes),
        )

    def encode(self, x):
        return self.features(x).squeeze(-1)

    def forward(self, x):
        return self.classifier(self.features(x))


class TemporalBiGRU(nn.Module):
    def __init__(self, input_dim=128, hidden_dim=64, n_classes=5):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(0.30)
        self.classifier = nn.Linear(
            hidden_dim * 2,
            n_classes,
        )

    def forward(self, x):
        sequence_output, _ = self.gru(x)
        center = x.shape[1] // 2
        center_output = sequence_output[:, center, :]

        return self.classifier(
            self.dropout(center_output)
        )

class EndToEndSleepModel(nn.Module):
    def __init__(self, cnn, temporal_head):
        super().__init__()

        self.encoder = cnn.features
        self.gru = temporal_head.gru
        self.dropout = temporal_head.dropout
        self.classifier = temporal_head.classifier

    def forward(self, x):
        batch_size, context_size, n_channels, n_samples = x.shape

        x = x.reshape(
            batch_size * context_size,
            n_channels,
            n_samples,
        )

        embeddings = self.encoder(x).squeeze(-1)

        embeddings = embeddings.reshape(
            batch_size,
            context_size,
            -1,
        )

        sequence_output, _ = self.gru(embeddings)

        center = context_size // 2
        center_output = sequence_output[:, center, :]

        return self.classifier(
            self.dropout(center_output)
        )


In [13]:
def run_finetuning(seed, signals, channel_mean, channel_std):
    reset_seed(seed)
    parts = [partition_index(indexes[5], subjects) for subjects in (train_subjects, val_subjects, test_subjects)]
    datasets = [TemporalSignalDataset(signals, p, channel_mean, channel_std) for p in parts]
    generator = torch.Generator().manual_seed(seed)
    loaders = [DataLoader(ds, batch_size=32, shuffle=(i == 0), generator=generator if i == 0 else None, num_workers=0) for i, ds in enumerate(datasets)]
    train_loader, val_loader, test_loader = loaders
    cnn = SleepCNN(n_channels=signals.shape[1])
    cnn.load_state_dict(torch.load(Path("../models/dl_cnn_baseline.pt"), map_location="cpu"))
    head = TemporalBiGRU()
    head.load_state_dict(torch.load(MODELS_DIR / f"dl_context_5_seed_{seed}_bigru.pt", map_location="cpu"))
    model = EndToEndSleepModel(cnn, head).to(DEVICE)
    frozen_val = evaluate_loader(model, val_loader)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best, stale, history = -np.inf, 0, []
    path = MODELS_DIR / f"finetuned_seed_{seed}.pt"
    for epoch in range(1, 13):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
            loss.backward()
            optimizer.step()
        metrics = evaluate_loader(model, val_loader)
        history.append({"epoch": epoch, **metrics})
        print("Fine-tuning", seed, epoch, metrics["macro_f1"])
        if metrics["macro_f1"] > best:
            best, stale = metrics["macro_f1"], 0
            torch.save(model.state_dict(), path)
        else: stale += 1
        if stale >= 3: break
    pd.DataFrame(history).to_csv(RESULTS_DIR / f"finetuning_history_seed_{seed}.csv", index=False)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    tuned_val = evaluate_loader(model, val_loader)
    tuned_test = evaluate_loader(model, test_loader)
    yt, yp = predict_loader(model, test_loader)
    pred = parts[2][["subject", "source_epoch_index"]].copy()
    pred["y_true"], pred["y_pred"] = yt, yp
    pred.to_parquet(RESULTS_DIR / f"finetuned_seed_{seed}_predictions.parquet", index=False)
    result = {"seed": seed, **{f"frozen_val_{k}": v for k,v in frozen_val.items()}, **{f"finetuned_val_{k}": v for k,v in tuned_val.items()}, **{f"finetuned_test_{k}": v for k,v in tuned_test.items()}}
    del model, cnn, head, optimizer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

if RUN_FINETUNING:
    signals = np.load(DATA_DIR / "dl_signals.npy", mmap_mode="r")
    if len(signals) != len(metadata): raise ValueError("Señales y metadata no coinciden")
    norm = json.loads(Path("../reports/results/dl_channel_normalization.json").read_text())
    fine_rows = []
    for seed in SEEDS:
        fine_rows.append(run_finetuning(seed, signals, norm["channel_mean_uv"], norm["channel_std_uv"]))
        pd.DataFrame(fine_rows).to_csv(RESULTS_DIR / "finetuning_runs.csv", index=False)
    fine = pd.DataFrame(fine_rows).merge(test_runs.query("context_size == 5")[["seed", "macro_f1", "cohen_kappa"]].rename(columns={"macro_f1": "frozen_test_macro_f1", "cohen_kappa": "frozen_test_cohen_kappa"}), on="seed", validate="one_to_one")
    fine["delta_val_macro_f1"] = fine["finetuned_val_macro_f1"] - fine["frozen_val_macro_f1"]
    fine["delta_test_macro_f1"] = fine["finetuned_test_macro_f1"] - fine["frozen_test_macro_f1"]
    fine.to_csv(RESULTS_DIR / "finetuning_paired.csv", index=False)
    summary = fine.drop(columns="seed").agg(["mean", "std"])
    summary.to_csv(RESULTS_DIR / "finetuning_summary.csv")
    display(summary)
else:
    print("Fine-tuning multisemilla pendiente; no concluir robustez de esta comparación.")


Fine-tuning 42 1 0.7784679260190674
Fine-tuning 42 2 0.7767281713433418
Fine-tuning 42 3 0.7739978489982853
Fine-tuning 42 4 0.7757825525091195
Fine-tuning 123 1 0.7752810295909668
Fine-tuning 123 2 0.770021528896416
Fine-tuning 123 3 0.7780334118397519
Fine-tuning 123 4 0.760816625908231
Fine-tuning 123 5 0.7540822562532852
Fine-tuning 123 6 0.7659379386348912
Fine-tuning 2026 1 0.7776101772241548
Fine-tuning 2026 2 0.7642108722822105
Fine-tuning 2026 3 0.7671783520187844
Fine-tuning 2026 4 0.7755155722204381
Fine-tuning 7 1 0.7773425188656757
Fine-tuning 7 2 0.757386787304797
Fine-tuning 7 3 0.7734114220718088
Fine-tuning 7 4 0.772911406006733
Fine-tuning 99 1 0.7679804037930931
Fine-tuning 99 2 0.7688644388097824
Fine-tuning 99 3 0.7693034885964062
Fine-tuning 99 4 0.7795495145497784
Fine-tuning 99 5 0.7604243967515761
Fine-tuning 99 6 0.7605194103570463
Fine-tuning 99 7 0.7671569462736418


,frozen_val_cohen_kappa,frozen_val_accuracy,frozen_val_balanced_accuracy,frozen_val_macro_f1,finetuned_val_cohen_kappa,finetuned_val_accuracy,finetuned_val_balanced_accuracy,finetuned_val_macro_f1,finetuned_test_cohen_kappa,finetuned_test_accuracy,finetuned_test_balanced_accuracy,finetuned_test_macro_f1,frozen_test_macro_f1,frozen_test_cohen_kappa,delta_val_macro_f1,delta_test_macro_f1
mean,0.732207,0.793445,0.78985,0.778815,0.733842,0.795194,0.787366,0.778201,0.711950,0.778568,0.775862,0.762251,0.769603,0.719579,-0.000614,-0.007352
std,0.003086,0.002717,0.00137,0.002640,0.001376,0.001497,0.004292,0.000867,0.006923,0.005977,0.006047,0.005193,0.003285,0.004015,0.003272,0.005524


## Evaluación final de contexto 7 — sin entrenamiento

Bloque independiente del kernel anterior: carga las cinco predicciones y la CNN; valida etiquetas, claves únicas, cobertura y el mismo holdout. Guarda tablas globales, F1 por etapa, métricas por sujeto y pruebas en `reports/results/robustness/final_context7/`. Actualiza `reports/results/final_model_selection.json`.

Se presenta la media y el desvío entre semillas; la CNN es fija. El contraste complementario principal usa **30 diferencias**: para cada sujeto se promedia su Macro-F1 temporal entre semillas y se resta el de la CNN. No es un ensemble ni se combinan las 150 filas como sujetos independientes. Se incluyen los cinco Wilcoxon por semilla con ajuste Holm. El contraste de localización de Wilcoxon presupone diferencias aproximadamente simétricas; un p no significativo no prueba equivalencia. No se realiza aquí un contraste 7 frente a 5.


In [14]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score
from scipy.stats import wilcoxon, rankdata

# Esta sección es independiente: ejecutar desde aquí con kernel nuevo.
RESULTS = Path('../reports/results')
ROBUST = RESULTS / 'robustness'
OUT = ROBUST / 'final_context7'
SEEDS = [42, 123, 2026, 7, 99]
LABELS = [0, 1, 2, 3, 4]
STAGES = ['W', 'N1', 'N2', 'N3', 'REM']
KEYS = ['subject', 'source_epoch_index']
paths = [RESULTS / 'dl_cnn_holdout_predictions.parquet', ROBUST / 'validation_runs.csv']
paths += [ROBUST / f'context_7_seed_{seed}_predictions.parquet' for seed in SEEDS]
missing = [str(p) for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan predicciones/resultados guardados; no reentrenar:\n' + '\n'.join(missing))

validation = pd.read_csv(ROBUST / 'validation_runs.csv')
if validation.duplicated(['context_size', 'seed']).any():
    raise ValueError('Hay corridas de validation duplicadas')
for context in [3, 5, 7]:
    if set(validation.loc[validation.context_size == context, 'seed']) != set(SEEDS):
        raise ValueError('Se requieren las cinco semillas en cada contexto')
selected = int(validation.groupby('context_size').val_macro_f1.mean().idxmax())
if selected != 7:
    raise ValueError(f'Validation selecciona {selected}; revisar antes de evaluar contexto 7 como final')

def clean_prediction(df, name):
    cols = KEYS + ['y_true', 'y_pred']
    if not set(cols).issubset(df.columns): raise ValueError(f'Faltan columnas en {name}')
    df = df[cols].copy()
    if df.isna().any().any() or df.duplicated(KEYS).any():
        raise ValueError(f'Nulos o claves duplicadas en {name}')
    if not df[['y_true', 'y_pred']].isin(LABELS).all().all():
        raise ValueError(f'Etiquetas inválidas en {name}')
    return df.sort_values(KEYS).reset_index(drop=True)

def scores(y, pred):
    return dict(accuracy=accuracy_score(y, pred),
                balanced_accuracy=balanced_accuracy_score(y, pred),
                macro_f1=f1_score(y, pred, labels=LABELS, average='macro', zero_division=0),
                cohen_kappa=cohen_kappa_score(y, pred, labels=LABELS))

def paired_test(delta):
    d = np.round(np.asarray(delta, dtype=float), 12)
    if not len(d) or not np.isfinite(d).all(): raise ValueError('Diferencias inválidas')
    nz = d[d != 0]
    if len(nz):
        test = wilcoxon(d, zero_method='wilcox', alternative='two-sided', method='auto')
        ranks = rankdata(np.abs(nz))
        w, p, effect = float(test.statistic), float(test.pvalue), float(np.sum(ranks*np.sign(nz))/ranks.sum())
    else: w, p, effect = 0., 1., 0.
    return dict(n_subjects=len(d), improved=int((d>0).sum()), worse=int((d<0).sum()),
                tied=int((d==0).sum()), mean_delta=float(d.mean()), median_delta=float(np.median(d)),
                statistic=w, p_two_sided=p, rank_biserial=effect)

cnn = clean_prediction(pd.read_parquet(paths[0]), 'CNN')
global_rows, stage_rows, subject_rows, test_rows = [], [], [], []
reference = None
for seed in SEEDS:
    temporal = clean_prediction(pd.read_parquet(ROBUST / f'context_7_seed_{seed}_predictions.parquet'), str(seed))
    signature = temporal[KEYS + ['y_true']]
    if reference is None: reference = signature.copy()
    elif not signature.equals(reference): raise ValueError('Las semillas no comparten epochs y etiquetas')
    merged = temporal.merge(cnn, on=KEYS, how='left', suffixes=('_temporal','_cnn'), validate='one_to_one')
    if merged.isna().any().any() or not np.array_equal(merged.y_true_temporal, merged.y_true_cnn):
        raise ValueError('La CNN no cubre todos los epochs o las etiquetas no coinciden')
    if merged.subject.nunique() != 30 or len(merged) != 26446:
        raise ValueError('Cambió el holdout esperado (30 sujetos / 26446 epochs); revisar archivos')
    for model in ['cnn', 'temporal']:
        y, pred = merged.y_true_temporal, merged[f'y_pred_{model}']
        global_rows.append(dict(seed=seed, model=model, n_epochs=len(merged), **scores(y, pred)))
        values = f1_score(y, pred, labels=LABELS, average=None, zero_division=0)
        for label, stage, value in zip(LABELS, STAGES, values):
            stage_rows.append(dict(seed=seed, model=model, stage=stage, f1=value, support=int((y==label).sum())))
    for subject, group in merged.groupby('subject'):
        baseline = scores(group.y_true_temporal, group.y_pred_cnn)
        temporal_metrics = scores(group.y_true_temporal, group.y_pred_temporal)
        subject_rows.append(dict(seed=seed, subject=subject, n_epochs=len(group),
                                 cnn_macro_f1=baseline['macro_f1'], temporal_macro_f1=temporal_metrics['macro_f1'],
                                 delta_macro_f1=temporal_metrics['macro_f1']-baseline['macro_f1'],
                                 cnn_kappa=baseline['cohen_kappa'], temporal_kappa=temporal_metrics['cohen_kappa']))
    seed_rows = [r for r in subject_rows if r['seed'] == seed]
    test_rows.append(dict(seed=seed, **paired_test([r['delta_macro_f1'] for r in seed_rows])))

global_runs = pd.DataFrame(global_rows)
per_stage = pd.DataFrame(stage_rows)
per_subject = pd.DataFrame(subject_rows)
metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'cohen_kappa']
global_summary = global_runs.groupby('model')[metrics].agg(['mean','std'])
stage_summary = per_stage.groupby(['stage','model']).f1.agg(['mean','std']).reset_index()
subject_summary = per_subject.groupby('subject').agg(
    n_epochs=('n_epochs','first'), cnn_macro_f1=('cnn_macro_f1','first'),
    temporal_macro_f1_mean=('temporal_macro_f1','mean'), temporal_macro_f1_std=('temporal_macro_f1','std'),
    delta_mean=('delta_macro_f1','mean'), delta_std=('delta_macro_f1','std'),
    improved_seeds=('delta_macro_f1',lambda s: int((s>0).sum())))
# Un único par por sujeto: promedio de métricas entre semillas, NO ensemble de predicciones.
aggregate_test = paired_test(subject_summary.delta_mean)
seed_tests = pd.DataFrame(test_rows)
# Cinco contrastes complementarios; ajuste Holm para controlar multiplicidad.
order = np.argsort(seed_tests.p_two_sided.to_numpy())
adjusted = np.minimum(1., np.maximum.accumulate(seed_tests.p_two_sided.to_numpy()[order] * np.arange(len(SEEDS),0,-1)))
seed_tests['p_holm'] = 0.
seed_tests.loc[order, 'p_holm'] = adjusted

OUT.mkdir(parents=True, exist_ok=True)
for name, frame in [('global_runs',global_runs),('stage_runs',per_stage),('subject_runs',per_subject),('wilcoxon_per_seed',seed_tests)]:
    frame.to_csv(OUT / f'{name}.csv', index=False)
global_summary.to_csv(OUT / 'global_summary.csv')
stage_summary.to_csv(OUT / 'stage_summary.csv', index=False)
subject_summary.to_csv(OUT / 'subject_summary.csv')
(OUT / 'wilcoxon_subject_seed_mean.json').write_text(json.dumps(aggregate_test, indent=2))
selection = dict(selected_configuration='CNN congelada + BiGRU', context_size_epochs=7,
    seeds=SEEDS, selection_metric='mean_validation_macro_f1',
    validation_macro_f1_mean=float(validation.query('context_size == 7').val_macro_f1.mean()),
    validation_macro_f1_std=float(validation.query('context_size == 7').val_macro_f1.std()),
    checkpoint_policy='Todas las semillas; no se seleccionó un checkpoint por test',
    finetuning_scope='Solo contexto 5; no evaluado en contexto 7',
    inference_scope='Mismo encoder CNN y split; análisis complementario por sujeto, no 150 observaciones independientes')
(RESULTS / 'final_model_selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding='utf-8')
print('Resultados globales: media ± desvío entre semillas; CNN fija')
display(global_summary)
print('F1 por etapa:')
display(stage_summary)
print('Resultados por sujeto (cinco semillas, mismos 30 sujetos):')
display(subject_summary)
print('Wilcoxon sobre la diferencia media por sujeto (análisis complementario):')
display(pd.DataFrame([aggregate_test]))
print('Wilcoxon por semilla con ajuste Holm:')
display(seed_tests)
print('Selección final:', selection)


Resultados globales: media ± desvío entre semillas; CNN fija


accuracy           balanced_accuracy            macro_f1            \
              mean       std              mean       std      mean       std   
model                                                                          
cnn       0.742986  0.000000          0.752871  0.000000  0.734831  0.000000   
temporal  0.789102  0.002489          0.788695  0.003377  0.774928  0.001841   

         cohen_kappa            
                mean       std  
model                           
cnn         0.669235  0.000000  
temporal    0.725698  0.003195

F1 por etapa:


,stage,model,mean,std
0,N1,cnn,0.470746,0.000000
1,N1,temporal,0.517318,0.002320
2,N2,cnn,0.754018,0.000000
3,N2,temporal,0.803387,0.004543
4,N3,cnn,0.838204,0.000000
5,N3,temporal,0.855923,0.002866
6,REM,cnn,0.829468,0.000000
7,REM,temporal,0.870294,0.001661
8,W,cnn,0.781718,0.000000
9,W,temporal,0.827717,0.005409


Resultados por sujeto (cinco semillas, mismos 30 sujetos):


,n_epochs,cnn_macro_f1,temporal_macro_f1_mean,temporal_macro_f1_std,delta_mean,delta_std,improved_seeds
subject,,,,,,,
SN010,918,0.682980,0.735914,0.010694,0.052934,0.010694,5
SN013,998,0.737235,0.759177,0.006015,0.021941,0.006015,5
SN020,942,0.578732,0.642308,0.012836,0.063575,0.012836,5
SN021,950,0.672393,0.743327,0.006972,0.070934,0.006972,5
SN028,887,0.835744,0.856425,0.008287,0.020681,0.008287,5
SN031,767,0.668067,0.746226,0.015069,0.078159,0.015069,5
SN034,881,0.753245,0.781668,0.016708,0.028423,0.016708,5
SN035,990,0.555479,0.608020,0.007940,0.052541,0.007940,5
SN040,795,0.761686,0.808735,0.007256,0.047049,0.007256,5


Wilcoxon sobre la diferencia media por sujeto (análisis complementario):


,n_subjects,improved,worse,tied,mean_delta,median_delta,statistic,p_two_sided,rank_biserial
0,30,29,1,0,0.043765,0.046713,12.0,1.303852e-07,0.948387


Wilcoxon por semilla con ajuste Holm:


,seed,n_subjects,improved,worse,tied,mean_delta,median_delta,statistic,p_two_sided,rank_biserial,p_holm
0,42,30,29,1,0,0.042163,0.044804,16.0,3.147870e-07,0.931183,4.097819e-07
1,123,30,29,1,0,0.044427,0.040020,11.0,1.024455e-07,0.952688,4.097819e-07
2,2026,30,28,2,0,0.045698,0.045839,9.0,6.146729e-08,0.961290,3.073364e-07
3,7,30,29,1,0,0.042078,0.042829,11.0,1.024455e-07,0.952688,4.097819e-07
4,99,30,28,2,0,0.044456,0.047901,14.0,2.048910e-07,0.939785,4.097819e-07


Selección final: {'selected_configuration': 'CNN congelada + BiGRU', 'context_size_epochs': 7, 'seeds': [42, 123, 2026, 7, 99], 'selection_metric': 'mean_validation_macro_f1', 'validation_macro_f1_mean': 0.7814343370495204, 'validation_macro_f1_std': 0.0016721704003286708, 'checkpoint_policy': 'Todas las semillas; no se seleccionó un checkpoint por test', 'finetuning_scope': 'Solo contexto 5; no evaluado en contexto 7', 'inference_scope': 'Mismo encoder CNN y split; análisis complementario por sujeto, no 150 observaciones independientes'}


## Conclusiones finales

### Tamaño de contexto

Se compararon ventanas de 3, 5 y 7 epochs utilizando cinco semillas, con el mismo encoder CNN y la misma partición por sujeto.

| Contexto | Macro-F1 validation, media ± desvío | Macro-F1 test, media ± desvío |
|---|---|---|
| 3 | 0,7741 ± 0,0025 | 0,7667 ± 0,0023 |
| 5 | 0,7788 ± 0,0026 | 0,7696 ± 0,0033 |
| 7 | 0,7814 ± 0,0017 | 0,7749 ± 0,0018 |

Se selecciona CNN congelada + BiGRU con 7 epochs por la mayor media de Macro-F1 de validation. La diferencia pareada de validation entre 7 y 5 fue de 0,0026 ± 0,0035. Esta ventaja es pequeña y no se ha demostrado superioridad estadística ni equivalencia entre ambas ventanas.

La selección inicial de 5 epochs correspondía a una única corrida y queda documentada en NB11. La revisión multisemilla actualiza esa elección. Se reportan las cinco semillas, sin seleccionar un checkpoint por su desempeño en test.

### Comparación con la CNN por epoch

Sobre los mismos 26.446 epochs de 30 sujetos, el modelo temporal de 7 epochs alcanzó un Macro-F1 de 0,7749 ± 0,0018, frente a 0,7348 de la CNN. Cohen’s kappa pasó de 0,6692 a 0,7257 ± 0,0032.

El F1 promedio mejoró en las cinco etapas. N1 pasó de 0,4707 a 0,5173, aunque continúa siendo la etapa más difícil de clasificar.

Al promediar las cinco semillas, 29 de los 30 sujetos mejoraron respecto de la CNN. En 28 sujetos la mejora apareció en todas las corridas. SN081 mejoró en tres de cinco semillas y SN069 presentó una disminución en todas.

La diferencia media de Macro-F1 por sujeto fue de 0,0438. El análisis complementario de Wilcoxon sobre las diferencias medias por sujeto dio W = 12 y p ≈ 1,30 × 10⁻⁷, con una correlación biserial de rangos de 0,9484. Las comparaciones por semilla también resultaron significativas después del ajuste Holm.

Estos contrastes respaldan la mejora frente a la CNN por epoch; no evalúan la superioridad de la ventana 7 frente a la ventana 5.

### Fine-tuning

Con contexto 5, el fine-tuning obtuvo un Macro-F1 de validation de 0,7782 ± 0,0009, frente a 0,7788 ± 0,0026 con la CNN congelada. En test obtuvo 0,7623 ± 0,0052, frente a 0,7696 ± 0,0033.

El ajuste conjunto no aportó una mejora promedio según el criterio de validation. Esta conclusión se limita al contexto 5: no se evaluó fine-tuning con contexto 7.

### Alcance

Los resultados respaldan el uso de contexto temporal y la selección de la configuración congelada con 7 epochs. La variabilidad reportada corresponde a distintas semillas de la BiGRU, manteniendo fijo el encoder CNN y el split; no representa la variabilidad de reentrenar el pipeline completo.

La CNN de referencia es única. Las cinco semillas se evalúan sobre los mismos 30 sujetos y no constituyen 150 observaciones independientes. Además, la ventana centrada utiliza epochs futuros, por lo que el enfoque corresponde a clasificación offline.